# Week 3 – Unsupervised Learning & Model Evaluation

**Part A – Clustering and dimensionality reduction** on the Wine dataset (178 wines, 13 chemical measurements)
K-Means, Hierarchical Clustering, DBSCAN (extra), PCA

**Part B – Model evaluation** on the Breast Cancer Wisconsin dataset (569 tumours, 30 measurements)
Cross-validation, confusion matrix, precision, recall, F1-score, ROC-AUC, PR curve, hyperparameter tuning, threshold selection, PCA + classification

Both datasets are built into scikit-learn. All preprocessing is done inside pipelines so no information leaks from the test data.

In [1]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score,
                             accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, precision_recall_curve, classification_report)
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from scipy.cluster.hierarchy import linkage, dendrogram

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RS = 42
res = {}

# Part A: Clustering and dimensionality reduction (Wine dataset)

## A1. Load and look at the data
Clustering is **unsupervised**: the algorithm never sees the labels. The Wine dataset does come with the true grape variety (3 classes), which I will use only at the end to check how well the clusters match reality.

In [2]:
wine = load_wine(as_frame=True)
Xw, yw = wine.data, wine.target
print("Shape:", Xw.shape)
print("True class counts:", yw.value_counts().sort_index().to_dict())
print("Missing values:", int(Xw.isnull().sum().sum()))
Xw.describe().T[["mean","std","min","max"]].round(2)

Shape: (178, 13)
True class counts: {0: 59, 1: 71, 2: 48}
Missing values: 0


,mean,std,min,max
alcohol,13.00,0.81,11.03,14.83
malic_acid,2.34,1.12,0.74,5.80
ash,2.37,0.27,1.36,3.23
alcalinity_of_ash,19.49,3.34,10.60,30.00
magnesium,99.74,14.28,70.00,162.00
total_phenols,2.30,0.63,0.98,3.88
flavanoids,2.03,1.00,0.34,5.08
nonflavanoid_phenols,0.36,0.12,0.13,0.66
proanthocyanins,1.59,0.57,0.41,3.58
color_intensity,5.06,2.32,1.28,13.00


The features are on very different scales (proline is in the hundreds or thousands while hue is around 1), so distance-based clustering would be dominated by proline. I standardize the features first.

In [3]:
scaler = StandardScaler()
Xw_s = scaler.fit_transform(Xw)
print("Mean after scaling (first 3):", Xw_s.mean(axis=0)[:3].round(2), "| Std:", Xw_s.std(axis=0)[:3].round(2))

Mean after scaling (first 3): [ 0.  0. -0.] | Std: [1. 1. 1.]


## A2. Dimensionality reduction with PCA

In [4]:
pca_full = PCA().fit(Xw_s)
evr = pca_full.explained_variance_ratio_
cum = np.cumsum(evr)
n95 = int(np.argmax(cum >= 0.95) + 1)
print("Variance explained by PC1, PC2:", evr[:2].round(3), "| together:", round(cum[1],3))
print("Components needed for 95% variance:", n95, "of", Xw.shape[1])

fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].bar(range(1, 14), evr, color="#2E5597"); ax[0].set_title("Variance explained by each component")
ax[0].set_xlabel("Principal component"); ax[0].set_ylabel("Explained variance ratio")
ax[1].plot(range(1, 14), cum, marker="o", color="#2E5597"); ax[1].axhline(0.95, color="r", ls="--", label="95%")
ax[1].set_title("Cumulative explained variance"); ax[1].set_xlabel("Number of components"); ax[1].legend()
plt.tight_layout(); plt.savefig("figures/01_pca_variance.png", dpi=130); plt.show()
res["pca"] = {"pc1": float(evr[0]), "pc2": float(evr[1]), "two": float(cum[1]), "n95": n95}

Variance explained by PC1, PC2: [0.362 0.192] | together: 0.554
Components needed for 95% variance: 10 of 13


In [5]:
pca2 = PCA(n_components=2, random_state=RS)
Xw_2d = pca2.fit_transform(Xw_s)
plt.figure(figsize=(6,5))
sns.scatterplot(x=Xw_2d[:,0], y=Xw_2d[:,1], hue=yw.map({0:"class 0",1:"class 1",2:"class 2"}), palette="Set1", s=50)
plt.title("Wine data projected to 2 dimensions (true classes)")
plt.xlabel(f"PC1 ({evr[0]*100:.1f}%)"); plt.ylabel(f"PC2 ({evr[1]*100:.1f}%)")
plt.tight_layout(); plt.savefig("figures/02_pca_true_classes.png", dpi=130); plt.show()

**Observation:** the first two components keep about 55% of the information, and the three grape varieties already separate visibly. PCA is used here only to *draw* the clusters in 2D. The clustering itself runs on all 13 scaled features.

## A3. K-Means: choosing the number of clusters

In [6]:
ks = range(2, 11)
rows = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RS).fit(Xw_s)
    rows.append({"k": k, "Inertia": km.inertia_,
                 "Silhouette": silhouette_score(Xw_s, km.labels_),
                 "Davies-Bouldin": davies_bouldin_score(Xw_s, km.labels_),
                 "Calinski-Harabasz": calinski_harabasz_score(Xw_s, km.labels_)})
k_table = pd.DataFrame(rows).set_index("k").round(3)
k_table

,Inertia,Silhouette,Davies-Bouldin,Calinski-Harabasz
k,,,,
2,1658.759,0.259,1.526,69.523
3,1277.928,0.285,1.389,70.940
4,1175.428,0.260,1.797,56.181
5,1109.513,0.202,1.808,46.952
6,1046.002,0.237,1.554,41.701
7,981.595,0.204,1.661,38.686
8,935.201,0.157,1.741,35.805
9,889.893,0.150,1.708,33.807
10,845.895,0.144,1.707,32.397


In [7]:
fig, ax = plt.subplots(1, 3, figsize=(14,4))
ax[0].plot(k_table.index, k_table["Inertia"], marker="o"); ax[0].set_title("Elbow method (inertia)"); ax[0].set_xlabel("k")
ax[1].plot(k_table.index, k_table["Silhouette"], marker="o", color="green"); ax[1].set_title("Silhouette score (higher is better)"); ax[1].set_xlabel("k")
ax[2].plot(k_table.index, k_table["Davies-Bouldin"], marker="o", color="crimson"); ax[2].set_title("Davies-Bouldin (lower is better)"); ax[2].set_xlabel("k")
plt.tight_layout(); plt.savefig("figures/03_choose_k.png", dpi=130); plt.show()
print("Best k by silhouette:", k_table["Silhouette"].idxmax())

Best k by silhouette: 3


The elbow bends around k = 3 and the silhouette score is highest at k = 3, so I choose **k = 3**. The silhouette and Davies-Bouldin scores need no labels, so they are valid even when the real answer is unknown.

## A4. K-Means with k = 3

In [8]:
km = KMeans(n_clusters=3, n_init=10, random_state=RS).fit(Xw_s)
km_labels = km.labels_
km_res = {"Silhouette": silhouette_score(Xw_s, km_labels), "Davies-Bouldin": davies_bouldin_score(Xw_s, km_labels),
          "Calinski-Harabasz": calinski_harabasz_score(Xw_s, km_labels),
          "ARI vs true labels": adjusted_rand_score(yw, km_labels), "NMI vs true labels": normalized_mutual_info_score(yw, km_labels)}
print({k: round(v,3) for k,v in km_res.items()})
print()
print("Cluster sizes:", pd.Series(km_labels).value_counts().sort_index().to_dict())
pd.crosstab(yw.rename("True class"), pd.Series(km_labels, name="K-Means cluster"))

{'Silhouette': 0.285, 'Davies-Bouldin': 1.389, 'Calinski-Harabasz': 70.94, 'ARI vs true labels': 0.897, 'NMI vs true labels': 0.876}

Cluster sizes: {0: 65, 1: 51, 2: 62}


K-Means cluster,0,1,2
True class,,,
0,0,0,59
1,65,3,3
2,0,48,0


In [9]:
centers_2d = pca2.transform(km.cluster_centers_)
fig, ax = plt.subplots(1, 2, figsize=(12,5))
sns.scatterplot(x=Xw_2d[:,0], y=Xw_2d[:,1], hue=yw, palette="Set1", s=50, ax=ax[0]); ax[0].set_title("True classes")
sns.scatterplot(x=Xw_2d[:,0], y=Xw_2d[:,1], hue=km_labels, palette="Set2", s=50, ax=ax[1])
ax[1].scatter(centers_2d[:,0], centers_2d[:,1], c="black", marker="X", s=220, label="Centroids"); ax[1].set_title("K-Means clusters (k=3)"); ax[1].legend()
for a in ax: a.set_xlabel("PC1"); a.set_ylabel("PC2")
plt.tight_layout(); plt.savefig("figures/04_kmeans_clusters.png", dpi=130); plt.show()

### Silhouette plot (how well does each point fit its own cluster?)

In [10]:
from sklearn.metrics import silhouette_samples
sil = silhouette_samples(Xw_s, km_labels)
plt.figure(figsize=(7,4.5)); y_low = 10
for c in range(3):
    v = np.sort(sil[km_labels == c]); y_up = y_low + len(v)
    plt.fill_betweenx(np.arange(y_low, y_up), 0, v, alpha=0.8, label=f"Cluster {c}"); y_low = y_up + 10
plt.axvline(sil.mean(), color="red", ls="--", label=f"Mean = {sil.mean():.2f}")
plt.xlabel("Silhouette coefficient"); plt.yticks([]); plt.title("Silhouette plot for K-Means (k=3)"); plt.legend()
plt.tight_layout(); plt.savefig("figures/05_silhouette_plot.png", dpi=130); plt.show()
print("Points with negative silhouette:", int((sil < 0).sum()))

Points with negative silhouette: 7


### Does scaling really matter? K-Means on raw vs scaled data

In [11]:
km_raw = KMeans(n_clusters=3, n_init=10, random_state=RS).fit(Xw)
raw_ari = adjusted_rand_score(yw, km_raw.labels_)
print("ARI without scaling:", round(raw_ari,3), "| with scaling:", round(km_res["ARI vs true labels"],3))

ARI without scaling: 0.371 | with scaling: 0.897


### What do the clusters mean? (average of original features per cluster)

In [12]:
prof = Xw.copy(); prof["cluster"] = km_labels
profile = prof.groupby("cluster")[["alcohol","flavanoids","color_intensity","proline","total_phenols"]].mean().round(2)
profile["size"] = prof["cluster"].value_counts().sort_index()
profile

,alcohol,flavanoids,color_intensity,proline,total_phenols,size
cluster,,,,,,
0,12.25,2.05,2.97,510.17,2.25,65
1,13.13,0.82,7.23,619.06,1.68,51
2,13.68,3.00,5.45,1100.23,2.85,62


## A5. Hierarchical clustering

In [13]:
Z = linkage(Xw_s, method="ward")
plt.figure(figsize=(11,4.5))
dendrogram(Z, no_labels=True, color_threshold=Z[-3,2]+0.01, above_threshold_color="grey")
plt.axhline(Z[-3,2]+0.01, color="red", ls="--", label="Cut for 3 clusters")
plt.title("Dendrogram (Ward linkage)"); plt.ylabel("Distance"); plt.legend()
plt.tight_layout(); plt.savefig("figures/06_dendrogram.png", dpi=130); plt.show()

In [14]:
hrows = []
for link in ["ward", "complete", "average", "single"]:
    lab = AgglomerativeClustering(n_clusters=3, linkage=link).fit_predict(Xw_s)
    hrows.append({"Linkage": link, "Silhouette": silhouette_score(Xw_s, lab),
                  "ARI vs true labels": adjusted_rand_score(yw, lab), "NMI vs true labels": normalized_mutual_info_score(yw, lab),
                  "Cluster sizes": sorted(pd.Series(lab).value_counts().tolist(), reverse=True)})
h_table = pd.DataFrame(hrows).set_index("Linkage"); h_table[["Silhouette","ARI vs true labels","NMI vs true labels"]] = h_table[["Silhouette","ARI vs true labels","NMI vs true labels"]].round(3)
h_table

,Silhouette,ARI vs true labels,NMI vs true labels,Cluster sizes
Linkage,,,,
ward,0.277,0.790,0.786,"[64, 58, 56]"
complete,0.204,0.577,0.614,"[69, 58, 51]"
average,0.158,-0.005,0.018,"[174, 3, 1]"
single,0.183,-0.007,0.035,"[174, 3, 1]"


Ward linkage minimises the variance inside each cluster and usually gives balanced clusters. Single linkage tends to chain points together and produce one huge cluster and a few tiny ones, which is what the cluster sizes show.

In [15]:
ward_labels = AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(Xw_s)
pd.crosstab(yw.rename("True class"), pd.Series(ward_labels, name="Ward cluster"))

Ward cluster,0,1,2
True class,,,
0,0,0,59
1,58,8,5
2,0,48,0


## A6. DBSCAN (density-based, extra)

In [16]:
drows = []
for eps in [1.8, 2.0, 2.2, 2.5, 3.0]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xw_s)
    ncl = len(set(lab)) - (1 if -1 in lab else 0)
    mask = lab != -1
    sil_d = silhouette_score(Xw_s[mask], lab[mask]) if ncl >= 2 else np.nan
    drows.append({"eps": eps, "Clusters found": ncl, "Noise points": int((lab==-1).sum()),
                  "Silhouette (no noise)": sil_d, "ARI vs true labels": adjusted_rand_score(yw, lab)})
d_table = pd.DataFrame(drows).set_index("eps").round(3)
d_table

,Clusters found,Noise points,Silhouette (no noise),ARI vs true labels
eps,,,,
1.8,7,118,0.246,0.084
2.0,5,85,0.240,0.220
2.2,2,55,0.348,0.329
2.5,1,24,NaN,0.005
3.0,1,11,NaN,-0.006


DBSCAN needs a good `eps`, and in 13 dimensions the points are spread out so distances are all similar. It finds either too few clusters or marks many points as noise, so K-Means and Ward are the better choices for this dataset.

## A7. Clustering comparison

In [17]:
comp = pd.DataFrame({
  "K-Means (k=3)": {"Silhouette": silhouette_score(Xw_s, km_labels), "ARI": adjusted_rand_score(yw, km_labels), "NMI": normalized_mutual_info_score(yw, km_labels)},
  "Hierarchical (Ward)": {"Silhouette": silhouette_score(Xw_s, ward_labels), "ARI": adjusted_rand_score(yw, ward_labels), "NMI": normalized_mutual_info_score(yw, ward_labels)},
}).T.round(3)
comp

,Silhouette,ARI,NMI
K-Means (k=3),0.285,0.897,0.876
Hierarchical (Ward),0.277,0.790,0.786


In [18]:
res["wine"] = {
  "k_table": k_table.reset_index().to_dict("records"),
  "km": km_res, "raw_ari": raw_ari,
  "km_sizes": pd.Series(km_labels).value_counts().sort_index().tolist(),
  "km_cross": pd.crosstab(yw, km_labels).values.tolist(),
  "neg_sil": int((sil < 0).sum()),
  "profile": profile.reset_index().to_dict("records"),
  "hier": h_table.reset_index().to_dict("records"),
  "ward_cross": pd.crosstab(yw, ward_labels).values.tolist(),
  "dbscan": d_table.reset_index().to_dict("records"),
  "comp": comp.reset_index().to_dict("records"),
}

# Part B: Model evaluation (Breast Cancer dataset)

## B1. Data and set-up
569 tumours, 30 numeric measurements. I treat **malignant as the positive class (1)** because missing a cancer (false negative) is the costly mistake. In scikit-learn the original coding is 0 = malignant, so I flip it.

In [19]:
bc = load_breast_cancer(as_frame=True)
X = bc.data
y = 1 - bc.target          # 1 = malignant, 0 = benign
print("Shape:", X.shape, "| Missing:", int(X.isnull().sum().sum()))
print("Benign:", int((y==0).sum()), "| Malignant:", int((y==1).sum()), f"({y.mean()*100:.1f}% malignant)")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Shape: (569, 30) | Missing: 0
Benign: 357 | Malignant: 212 (37.3% malignant)
Train: (455, 30) | Test: (114, 30)


In [20]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=RS)),
    "KNN":                 make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM (RBF)":           make_pipeline(StandardScaler(), SVC(probability=True, random_state=RS)),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=RS),
}
print("Every model that needs scaling has the scaler inside its pipeline, so scaling is learned from training folds only.")

Every model that needs scaling has the scaler inside its pipeline, so scaling is learned from training folds only.


## B2. Cross-validation (5-fold, stratified, training data only)

In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
scoring = {"accuracy":"accuracy", "precision":"precision", "recall":"recall", "f1":"f1", "roc_auc":"roc_auc"}
cv_rows = []
for name, m in models.items():
    s = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring)
    row = {"Model": name}
    for k in scoring: row[k] = f"{s['test_'+k].mean():.3f} ± {s['test_'+k].std():.3f}"
    row["_f1"] = s["test_f1"].mean(); row["_auc"] = s["test_roc_auc"].mean()
    cv_rows.append(row)
cv_df = pd.DataFrame(cv_rows).set_index("Model")
cv_df.drop(columns=["_f1","_auc"])

,accuracy,precision,recall,f1,roc_auc
Model,,,,,
Logistic Regression,0.974 ± 0.015,0.977 ± 0.028,0.953 ± 0.040,0.964 ± 0.021,0.996 ± 0.005
KNN,0.965 ± 0.019,0.988 ± 0.025,0.918 ± 0.034,0.951 ± 0.026,0.987 ± 0.014
SVM (RBF),0.971 ± 0.005,0.977 ± 0.021,0.947 ± 0.034,0.961 ± 0.008,0.995 ± 0.005
Random Forest,0.963 ± 0.015,0.964 ± 0.011,0.935 ± 0.039,0.949 ± 0.021,0.989 ± 0.007


## B3. Test set evaluation

In [22]:
test_rows, fitted, preds, probs = [], {}, {}, {}
for name, m in models.items():
    m.fit(X_train, y_train); fitted[name] = m
    p = m.predict(X_test); pr = m.predict_proba(X_test)[:,1]; preds[name], probs[name] = p, pr
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    test_rows.append({"Model": name, "Accuracy": accuracy_score(y_test,p), "Precision": precision_score(y_test,p),
                      "Recall": recall_score(y_test,p), "F1": f1_score(y_test,p), "ROC-AUC": roc_auc_score(y_test,pr),
                      "PR-AUC": average_precision_score(y_test,pr), "TN":int(tn), "FP":int(fp), "FN":int(fn), "TP":int(tp)})
test_df = pd.DataFrame(test_rows).set_index("Model")
test_df.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,TN,FP,FN,TP
Model,,,,,,,,,,
Logistic Regression,0.965,0.975,0.929,0.951,0.996,0.994,71,1,3,39
KNN,0.956,0.974,0.905,0.938,0.982,0.973,71,1,4,38
SVM (RBF),0.974,1.000,0.929,0.963,0.995,0.993,72,0,3,39
Random Forest,0.965,1.000,0.905,0.950,0.994,0.991,72,0,4,38


**Metrics in plain words (positive = malignant)**
- **Precision:** of the tumours predicted malignant, how many really were
- **Recall:** of the truly malignant tumours, how many we caught (the most important one here)
- **F1:** balance of precision and recall
- **ROC-AUC:** how well the model ranks malignant above benign across all thresholds
- **PR-AUC:** like ROC-AUC but focused on the positive class

In [23]:
fig, axes = plt.subplots(1, 4, figsize=(16,3.8))
for ax, (n, p) in zip(axes, preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, p), display_labels=["Benign","Malignant"]).plot(ax=ax, colorbar=False, cmap="Blues"); ax.set_title(n)
plt.tight_layout(); plt.savefig("figures/07_confusion_matrices.png", dpi=130); plt.show()

In [24]:
fig, ax = plt.subplots(1, 2, figsize=(12,4.8))
for n, pr in probs.items():
    fpr, tpr, _ = roc_curve(y_test, pr); ax[0].plot(fpr, tpr, label=f"{n} (AUC={roc_auc_score(y_test,pr):.3f})")
    pp, rr, _ = precision_recall_curve(y_test, pr); ax[1].plot(rr, pp, label=f"{n} (AP={average_precision_score(y_test,pr):.3f})")
ax[0].plot([0,1],[0,1],"k--"); ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate (recall)"); ax[0].set_title("ROC curves"); ax[0].legend(fontsize=8, loc="lower right")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-Recall curves"); ax[1].legend(fontsize=8, loc="lower left"); ax[1].set_ylim(0.8,1.01)
plt.tight_layout(); plt.savefig("figures/08_roc_pr_curves.png", dpi=130); plt.show()

In [25]:
best_cv_name = cv_df["_f1"].idxmax()
print("Classification report for", best_cv_name, "(best CV F1):")
print(classification_report(y_test, preds[best_cv_name], target_names=["Benign","Malignant"], digits=3))

Classification report for Logistic Regression (best CV F1):
              precision    recall  f1-score   support

      Benign      0.959     0.986     0.973        72
   Malignant      0.975     0.929     0.951        42

    accuracy                          0.965       114
   macro avg      0.967     0.957     0.962       114
weighted avg      0.965     0.965     0.965       114



## B4. Hyperparameter tuning

I tune each model with cross-validation on the **training data only**, optimising F1. Grid search tries every combination; randomized search tries a random sample and is faster for big grids (used for Random Forest).

In [26]:
searches = {
  "Logistic Regression": GridSearchCV(models["Logistic Regression"], {"logisticregression__C":[0.01,0.1,1,10,100], "logisticregression__penalty":["l2"]}, cv=cv, scoring="f1", n_jobs=-1),
  "KNN": GridSearchCV(models["KNN"], {"kneighborsclassifier__n_neighbors":list(range(3,22,2)), "kneighborsclassifier__weights":["uniform","distance"]}, cv=cv, scoring="f1", n_jobs=-1),
  "SVM (RBF)": GridSearchCV(models["SVM (RBF)"], {"svc__C":[0.1,1,10,100], "svc__gamma":["scale",0.001,0.01,0.1]}, cv=cv, scoring="f1", n_jobs=-1),
  "Random Forest": RandomizedSearchCV(models["Random Forest"], {"n_estimators":[100,200,300,500], "max_depth":[3,5,8,12,None], "min_samples_leaf":[1,2,4], "max_features":["sqrt","log2",0.5]}, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=RS),
}
tune_rows, tuned = [], {}
for name, s in searches.items():
    s.fit(X_train, y_train); tuned[name] = s.best_estimator_
    p0 = preds[name]; p1 = s.best_estimator_.predict(X_test)
    pr1 = s.best_estimator_.predict_proba(X_test)[:,1]
    tune_rows.append({"Model": name, "Best params": {k.split("__")[-1]: v for k,v in s.best_params_.items()},
                      "CV F1 (tuned)": s.best_score_, "Test F1 before": f1_score(y_test,p0), "Test F1 after": f1_score(y_test,p1),
                      "Test recall before": recall_score(y_test,p0), "Test recall after": recall_score(y_test,p1),
                      "Test ROC-AUC after": roc_auc_score(y_test,pr1)})
tune_df = pd.DataFrame(tune_rows).set_index("Model")
pd.set_option("display.max_colwidth", 80)
tune_df.round(3)

,Best params,CV F1 (tuned),Test F1 before,Test F1 after,Test recall before,Test recall after,Test ROC-AUC after
Model,,,,,,,
Logistic Regression,"{'C': 1, 'penalty': 'l2'}",0.964,0.951,0.951,0.929,0.929,0.996
KNN,"{'n_neighbors': 3, 'weights': 'uniform'}",0.963,0.938,0.911,0.905,0.857,0.982
SVM (RBF),"{'C': 10, 'gamma': 'scale'}",0.967,0.963,0.963,0.929,0.929,0.993
Random Forest,"{'n_estimators': 100, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth...",0.961,0.950,0.963,0.905,0.929,0.990


Tuning is judged on the untouched test set. If the 'after' numbers are about the same as 'before', the default settings were already good for this dataset.

## B5. Threshold selection (precision vs recall trade-off)

In [27]:
sel_name = max(tune_rows, key=lambda r: r["CV F1 (tuned)"])["Model"]
sel = tuned[sel_name]; pr_sel = sel.predict_proba(X_test)[:,1]
print("Model chosen by CV F1 (not by test score):", sel_name)
trows = []
for t in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    p = (pr_sel >= t).astype(int); tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    trows.append({"Threshold": t, "Precision": precision_score(y_test,p), "Recall": recall_score(y_test,p), "F1": f1_score(y_test,p), "Missed cancers (FN)": int(fn), "False alarms (FP)": int(fp)})
th_df = pd.DataFrame(trows).set_index("Threshold").round(3)
th_df

Model chosen by CV F1 (not by test score): SVM (RBF)


,Precision,Recall,F1,Missed cancers (FN),False alarms (FP)
Threshold,,,,,
0.1,0.820,0.976,0.891,1,9
0.2,0.891,0.976,0.932,1,5
0.3,0.952,0.952,0.952,2,2
0.4,0.976,0.952,0.964,2,1
0.5,1.000,0.929,0.963,3,0
0.6,1.000,0.929,0.963,3,0
0.7,1.000,0.929,0.963,3,0
0.8,1.000,0.833,0.909,7,0
0.9,1.000,0.786,0.880,9,0


In [28]:
fig, ax = plt.subplots(figsize=(7,4.5))
ax.plot(th_df.index, th_df["Precision"], marker="o", label="Precision"); ax.plot(th_df.index, th_df["Recall"], marker="o", label="Recall"); ax.plot(th_df.index, th_df["F1"], marker="o", label="F1")
ax.set_xlabel("Decision threshold"); ax.set_ylabel("Score"); ax.set_title(f"Threshold trade-off ({sel_name})"); ax.legend()
plt.tight_layout(); plt.savefig("figures/09_threshold_tradeoff.png", dpi=130); plt.show()

Lowering the threshold catches more cancers (higher recall) but creates more false alarms (lower precision). In a medical setting a lower threshold is often preferred, because a false alarm leads to a follow-up test while a missed cancer can be far worse.

## B6. Dimensionality reduction + classification (PCA in a pipeline)

In [29]:
comps = [1,2,3,5,7,10,15,20,30]
pca_scores = []
for n in comps:
    pipe = make_pipeline(StandardScaler(), PCA(n_components=n, random_state=RS), LogisticRegression(max_iter=2000, random_state=RS))
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")
    pca_scores.append({"Components": n, "CV F1": s.mean(), "CV std": s.std()})
pca_df = pd.DataFrame(pca_scores).set_index("Components").round(3)
ev = PCA().fit(StandardScaler().fit_transform(X_train)).explained_variance_ratio_.cumsum()
n95b = int(np.argmax(ev >= 0.95) + 1)
print("Components for 95% variance:", n95b, "of 30 | PC1+PC2 variance:", round(ev[1],3))
pca_df

Components for 95% variance: 10 of 30 | PC1+PC2 variance: 0.631


,CV F1,CV std
Components,,
1,0.874,0.066
2,0.928,0.013
3,0.944,0.023
5,0.958,0.021
7,0.961,0.015
10,0.967,0.024
15,0.964,0.020
20,0.961,0.026
30,0.964,0.021


In [30]:
plt.figure(figsize=(7,4.5))
plt.errorbar(pca_df.index, pca_df["CV F1"], yerr=pca_df["CV std"], marker="o", capsize=3)
plt.xlabel("Number of PCA components"); plt.ylabel("CV F1-score"); plt.title("Logistic Regression F1 vs number of PCA components")
plt.tight_layout(); plt.savefig("figures/10_pca_vs_f1.png", dpi=130); plt.show()

In [31]:
# Breast cancer data in 2D, coloured by diagnosis
Xs = StandardScaler().fit_transform(X); X2 = PCA(2, random_state=RS).fit_transform(Xs)
plt.figure(figsize=(6,5))
sns.scatterplot(x=X2[:,0], y=X2[:,1], hue=y.map({0:"Benign",1:"Malignant"}), palette={"Benign":"#2E5597","Malignant":"#C00000"}, s=35)
plt.title("Breast cancer data projected to 2D (PCA)"); plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout(); plt.savefig("figures/11_bc_pca_2d.png", dpi=130); plt.show()

## B7. Can clustering find the diagnosis without labels?

In [32]:
km_bc = KMeans(n_clusters=2, n_init=10, random_state=RS).fit(Xs)
bc_ari = adjusted_rand_score(y, km_bc.labels_); bc_sil = silhouette_score(Xs, km_bc.labels_)
ct = pd.crosstab(y.map({0:"Benign",1:"Malignant"}).rename("Diagnosis"), pd.Series(km_bc.labels_, name="K-Means cluster"))
print("ARI:", round(bc_ari,3), "| Silhouette:", round(bc_sil,3)); ct

ARI: 0.654 | Silhouette: 0.343


K-Means cluster,0,1
Diagnosis,,
Benign,339,18
Malignant,36,176


In [33]:
res["bc"] = {
  "n": [int(len(X)), int(X.shape[1]), int((y==0).sum()), int((y==1).sum()), int(len(X_train)), int(len(X_test))],
  "cv": cv_df.drop(columns=["_f1","_auc"]).reset_index().to_dict("records"),
  "test": test_df.reset_index().to_dict("records"),
  "best_cv_name": best_cv_name,
  "tune": [{**r, "Best params": {k:(v if isinstance(v,(str,type(None))) else float(v)) for k,v in r["Best params"].items()}} for r in tune_df.reset_index().to_dict("records")],
  "sel_name": sel_name,
  "thr": th_df.reset_index().to_dict("records"),
  "pca": pca_df.reset_index().to_dict("records"), "n95": n95b, "two_var": float(ev[1]),
  "bc_km": {"ari": float(bc_ari), "sil": float(bc_sil), "cross": ct.values.tolist()},
}
json.dump(res, open("results.json","w"), indent=2, default=str)
print("saved")

saved
